# RAG Fusion

## What This Notebook Covers

This notebook implements RAG Fusion, an advanced retrieval technique that solves one of the core weaknesses of standard single-query RAG systems.

In a standard RAG pipeline, one user question produces one retrieval query. The retrieved documents are only as good as the vocabulary and phrasing of that single query. If the user phrases the question differently from how the relevant information is written in the knowledge base, retrieval quality degrades significantly.

RAG Fusion addresses this by using an LLM to generate multiple semantically diverse sub-queries from the original user question. Each sub-query is sent to the retriever independently, producing multiple ranked document lists. Those lists are then merged and re-scored using Reciprocal Rank Fusion, a mathematically principled algorithm that promotes documents appearing near the top of multiple lists. The final merged and re-ranked document set is passed to the LLM as context for answer generation.

The notebook is organised into six categories.

Category 1 handles all installation and environment setup including the Google Gemini API key.

Category 2 loads multiple text documents from a local folder and combines them into one corpus.

Category 3 splits the corpus into chunks, embeds them using BGE embeddings, and stores them in ChromaDB.

Category 4 builds a standard single-query RAG chain as a baseline for comparison.

Category 5 implements the full RAG Fusion pipeline including the multi-query generator, Reciprocal Rank Fusion, and the final answer chain.

Category 6 contains the conclusion.

## Category 1  Installation and Environment Setup

### Why These Libraries Are Needed

langchain is the core framework providing document loaders, text splitters, prompt templates, chains, and the LCEL pipe syntax used throughout this notebook.

huggingface_hub and sentence_transformers provide access to the BGE embedding model that converts text chunks into dense vectors for semantic search.

pypdf is used internally by some LangChain loaders to parse PDF files, though this notebook uses text files.

google-generativeai and langchain-google-genai provide the interface to the Gemini 2.5 Flash model which acts as both the multi-query generator and the final answer generator.

chromadb is the local vector database that stores chunk embeddings on disk and supports fast nearest-neighbor search at query time.

unstructured is required by DirectoryLoader when loading plain text files without specifying an explicit loader class.

### Loading the Google API Key Securely

The Google API key is stored in a local .env file rather than hardcoded in the notebook. python-dotenv reads that file and injects the key as an environment variable. os.environ then makes it available to any library that reads GOOGLE_API_KEY from the environment automatically. This keeps the key out of the notebook file so it is safe to push to GitHub.

In [ ]:
# Install all required libraries
# Run this cell once before running any other cell in the notebook
!pip -q install langchain huggingface_hub tiktoken pypdf
!pip -q install google-generativeai chromadb
!pip -q install sentence_transformers

In [ ]:
!pip install -U langchain-community

In [ ]:
# Install the unstructured library required by DirectoryLoader for loading plain text files
!pip install unstructured

In [ ]:
# Install the LangChain Google GenAI integration package
# This provides ChatGoogleGenerativeAI which wraps the Gemini API as a LangChain LLM object
%pip install --upgrade --quiet langchain-google-genai

In [ ]:
import os
from dotenv import load_dotenv

# load_dotenv reads the .env file from the current working directory
# and injects every key-value pair as an OS environment variable
# The .env file should contain a line: GOOGLE_API_KEY=your_key_here
load_dotenv()

# Retrieve the Google API key from the environment
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Also write it back explicitly to os.environ so that any library
# that reads GOOGLE_API_KEY from the environment finds it regardless of how it was set
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

### Verifying the LangChain Version and Importing Core Modules

Printing the LangChain version is a quick sanity check that confirms the correct version was installed. Import errors in LangChain are often version-specific because the package has reorganised its submodules several times across major releases. Knowing the exact version installed makes it much easier to diagnose import failures by checking the release changelog.

In [ ]:
import langchain
print(langchain.__version__)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import langchain

### Helper Function for Display

wrap_text is a display utility that formats long LLM responses for readable terminal output. textwrap.fill wraps a single line at a given character width. Because LLM responses often contain their own newline characters for paragraph breaks, the function splits on those newlines first, wraps each resulting line independently, and then joins them back together. This preserves the original paragraph structure while preventing lines from overflowing the terminal width.

In [ ]:
import textwrap

def wrap_text(text, width=90):
    # Split the input text into lines based on newline characters
    # This preserves the paragraph structure that the LLM put into its response
    lines = text.split('\n')

    # Wrap each line individually at the specified character width
    # textwrap.fill handles word boundaries so words are never cut mid-character
    wrapped_lines = [textwrap.fill(line, width=width) for line in lines]

    # Join the individually wrapped lines back together using newline characters
    wrapped_text = '\n'.join(wrapped_lines)

    return wrapped_text

### Initialising the Gemini LLM

ChatGoogleGenerativeAI wraps the Google Gemini API as a standard LangChain chat model. The model used is gemini-2.5-flash which is Google's fast, cost-efficient flash model.

temperature=0 makes the model fully deterministic. At temperature 0 the model always picks the single highest-probability next token at each step rather than sampling from the distribution. For a RAG pipeline where factual grounding is the goal, deterministic output is preferred over creative variation.

The quick test invocation confirms the API key is valid and the model is reachable before running any of the heavier pipeline cells.

## Google

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialise the Gemini 2.5 Flash model as the LLM for this notebook
# temperature=0: fully deterministic output, always picks the highest-probability token
# This model is used for both multi-query generation and final answer generation
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Quick test to confirm the API key is valid and the model is reachable
result = llm.invoke("What is RAG Fusion?")
print(result.content)

## Category 2  Loading Documents From the Local Folder

### Why DirectoryLoader

The knowledge base for this notebook consists of multiple plain text files stored in one folder. Loading each file individually with separate TextLoader calls would require hardcoding every filename. DirectoryLoader solves this by scanning an entire directory and loading every file that matches a glob pattern.

glob='*.txt' instructs DirectoryLoader to find and load only files with the .txt extension, ignoring any PDFs, images, or other file types in the same folder.

show_progress=True displays a tqdm progress bar during loading which is useful when the folder contains many files and loading takes noticeable time.

### Why We Limit to 50 Documents

The full folder may contain more documents than are needed for demonstration. Loading all of them would increase embedding time proportionally. Slicing to the first 50 documents keeps the notebook fast to run while still providing enough diversity in the knowledge base to demonstrate RAG Fusion meaningfully.

### Building the Raw Text Corpus

The text splitter in Category 3 works on plain strings, not on LangChain Document objects. We therefore concatenate all page_content strings from the 50 loaded documents into one large raw_text string. The splitter then operates on this combined corpus and produces uniformly sized chunks regardless of which original document each piece came from.

## Load in Docs

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Path to the folder containing all the Singapore-topic text files
# Adjust this path to wherever your data folder is located
data_path = (r"F:\Advance RAG\Data\Multiple_text_data")

In [ ]:
%%time
# DirectoryLoader scans the folder and loads every file matching the glob pattern
# glob='*.txt': load only plain text files, ignore all other file types
# show_progress=True: display a progress bar so loading time is visible
# The %%time magic cell measures total wall time for the load operation
loader = DirectoryLoader(data_path, glob="*.txt", show_progress=True)
docs = loader.load()

In [ ]:
# Check the total number of documents loaded from the folder
len(docs)

In [ ]:
# Limit the document set to the first 50 files to keep embedding time manageable
# In a production system you would process all documents
docs = docs[:50]
len(docs)

In [ ]:
# Inspect the first Document object to see its structure
# Shows both page_content (the text) and metadata (source file path)
docs[0]

In [ ]:
from pprint import pprint

# Pretty-print the metadata dictionary of the first document
# Metadata contains the source file path which is useful for source attribution
pprint(docs[0].metadata)
print("\nContent:\n")
print(docs[0].page_content)

In [ ]:
# Concatenate all document texts into one raw string
# The text splitter in the next category works on a plain string, not Document objects
# Joining all docs first ensures the splitter produces uniformly-sized chunks
# regardless of which original file each piece of text came from
raw_text = ""

for i, doc in enumerate(docs):
    text = doc.page_content
    raw_text += text

In [ ]:
# Check the total character count of the combined corpus
# This tells us how much text the splitter will process
len(raw_text)

## Category 3  Chunking, BGE Embeddings, and ChromaDB Vector Store

### Why We Split Into Chunks

The raw corpus is one large string that is far too long for any embedding model to encode as a single unit. Most embedding models have a maximum sequence length of 512 tokens. More importantly, retrieving an entire corpus as one unit would provide no useful signal for ranking which parts are most relevant to a query.

RecursiveCharacterTextSplitter breaks the corpus into smaller overlapping pieces by trying separators in order: paragraph breaks, then sentence endings, then word boundaries, then individual characters. This recursive strategy preserves natural language structure better than a naive fixed-window cut.

chunk_size=500 means each chunk contains at most 500 characters of text.

chunk_overlap=100 means the last 100 characters of one chunk are repeated at the start of the next. This overlap ensures that a sentence straddling a chunk boundary still appears completely in at least one chunk.

length_function=len uses character count (not token count) to measure chunk size.

is_separator_regex=False treats separator strings as plain literals rather than regular expressions.

### BGE Embeddings

BAAI/bge-small-en-v1.5 is a small but highly capable English embedding model from the Beijing Academy of Artificial Intelligence, specifically optimised for retrieval tasks. The bge-small variant produces 384-dimensional vectors and is significantly faster and smaller than bge-large while retaining strong retrieval quality for English text.

normalize_embeddings=True divides each output vector by its L2 norm so all vectors have unit length. When vectors are unit-normalised, computing cosine similarity is equivalent to computing a simple dot product, which is faster and numerically more stable.

### ChromaDB Vector Store

Chroma.from_texts takes the list of plain text chunk strings, calls the embedding function on every chunk, and stores both the raw text and its embedding vector together in a local ChromaDB collection. persist_directory='./chroma_db' writes the index to disk so it survives kernel restarts without needing to re-embed everything.

## BGE Embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

# bge-small-en-v1.5 produces 384-dimensional vectors optimised for retrieval tasks
model_name = "BAAI/bge-small-en-v1.5"

# normalize_embeddings=True: unit-normalise all output vectors
# This makes cosine similarity equivalent to dot product for faster and more stable comparison
encode_kwargs = {'normalize_embeddings': True}

In [ ]:
# Instantiate the BGE embedding model
# Downloads model weights from HuggingFace Hub on the first run, uses local cache afterwards
# model_kwargs device cuda line is commented out so the model runs on CPU by default
# Uncomment it and set device to cuda if a GPU is available for faster embedding
embedding_function = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    # model_kwargs={'device': 'cuda'},
    encode_kwargs=encode_kwargs,
)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,           # Each chunk contains at most 500 characters
    chunk_overlap=100,        # Last 100 characters of one chunk repeat at the start of the next
    length_function=len,      # Measure chunk size by character count not token count
    is_separator_regex=False, # Treat separator strings as plain literals not regex patterns
)

In [ ]:
# Split the raw combined corpus into a list of plain text chunk strings
# split_text operates on a plain string and returns a list of strings
# (as opposed to split_documents which works on Document objects)
texts = text_splitter.split_text(raw_text)

In [ ]:
# Check how many chunks were produced from the full corpus
len(texts)

In [ ]:
# Inspect one chunk to verify the splitting produced sensible sized pieces
print(texts[4])

## Vector DB

In [ ]:
%%time
# Build the ChromaDB vector store from all text chunks
# from_texts embeds every chunk string using embedding_function and stores text plus vector
# persist_directory='./chroma_db': write the index to disk so it survives kernel restarts
# The %%time magic cell shows how long the embedding and indexing operation takes
db = Chroma.from_texts(
    texts,
    embedding_function,
    persist_directory="./chroma_db"
)

In [ ]:
# Test similarity search directly on the vector store before building any retriever
# This confirms ChromaDB is working and returning relevant chunks for a given query
# k=5 returns the 5 most similar chunks to the query
query = "Tell me about Universal Studios Singapore?"

results = db.similarity_search(query, k=5)

In [ ]:
from pprint import pprint

# Display the text content of each of the 5 retrieved chunks
# Verifies that the retrieved content is actually relevant to the query
for doc in results:
    pprint(doc.page_content)

## Setup a Retriever

In [ ]:
# as_retriever wraps the ChromaDB vector store behind the standard LangChain retriever interface
# This abstraction allows the vector store to be used inside any LangChain chain
# without the chain needing to know it is a ChromaDB instance specifically
# MMR search (search_type='mmr') and fetch_k can be added inside the search_kwargs dict
retriever = db.as_retriever()

# Quick test to confirm the retriever returns documents for the same query
retriever.invoke(query)

## Category 4  Baseline Standard Single-Query RAG Chain

### Why We Build a Baseline First

Before implementing RAG Fusion it is important to establish a standard single-query RAG chain as a baseline. Running the same questions through both the baseline and the RAG Fusion pipeline makes it possible to observe concretely how multi-query generation and Reciprocal Rank Fusion change the answer quality.

### How the Standard RAG Chain Works

The chain is built using LCEL pipe syntax. When the chain is invoked with a question string it executes four steps in sequence.

Step 1: RunnablePassthrough forwards the question string unchanged to both the context key and the question key. The retriever receives the question, embeds it, and fetches the most relevant chunks from ChromaDB.

Step 2: The context (retrieved chunk text) and the question are inserted into the ChatPromptTemplate to produce a formatted prompt that instructs the LLM to answer only from the provided context.

Step 3: The formatted prompt is sent to Gemini 2.5 Flash which generates the answer.

Step 4: StrOutputParser extracts the plain string from the LLM response object.

The fundamental limitation of this chain is that it sends exactly one query to the retriever. If the user's phrasing does not match the vocabulary in the stored chunks, relevant information may be missed entirely. RAG Fusion in Category 5 addresses this weakness.

## Chat chain

In [ ]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
# Define the prompt template for the standard RAG chain
# {context}: will be filled with the retrieved chunk texts concatenated together
# {question}: will be filled with the original user question
# The instruction to answer ONLY from the context prevents the LLM from hallucinating
# information that does not appear in the retrieved documents
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

In [ ]:
# Build the ChatPromptTemplate from the template string
# from_template parses the curly-brace placeholders and creates the corresponding
# input variables that the chain must supply when formatting the prompt
prompt = ChatPromptTemplate.from_template(template)
prompt

In [ ]:
# Print the raw template text to verify the placeholders are correctly positioned
print(prompt.messages[0].prompt.template)

In [ ]:
# Inspect the LLM object to confirm it is the Gemini model configured with temperature=0
llm

In [ ]:
# Build the standard single-query RAG chain using LCEL pipe syntax
# Step 1: RunnableParallel runs retriever and passthrough simultaneously
#   context key: retriever fetches chunks from ChromaDB using the question as the query
#   question key: RunnablePassthrough forwards the original question string unchanged
# Step 2: The dict with context and question fills the prompt template
# Step 3: The filled prompt is sent to Gemini for answer generation
# Step 4: StrOutputParser extracts the plain string from the LLM output object
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Test 1: Standard RAG chain on a factual question about Universal Studios Singapore
# wrap_text formats the response for readable terminal display
text_reply = chain.invoke("Tell me about Universal Studio Singapore")

print(wrap_text(text_reply))

In [ ]:
# Test 2: Standard RAG chain on a list-style question
# This type of question often reveals retrieval gaps in a single-query system
# because the answer requires information spread across multiple documents
text_reply = chain.invoke("What are 5 most beautiful rivers in Singapore?")

print(wrap_text(text_reply))

## Category 5  RAG Fusion Pipeline With Multi-Query Generation and Reciprocal Rank Fusion

### The Core Idea

Standard RAG sends one query to the retriever. The quality of retrieval is entirely dependent on whether the user's exact phrasing matches the vocabulary in the stored chunks. A user who asks about USS attractions might miss a chunk that discusses Universal Studios Singapore rides because the vocabulary differs even though the meaning is identical.

RAG Fusion breaks this dependency by asking the LLM to generate 4 semantically diverse sub-queries from the original question. Each sub-query emphasises a different aspect or phrasing of the same underlying information need. Each sub-query is then sent to the retriever independently, producing 4 separate ranked document lists.

Those 4 lists are then merged and re-scored using Reciprocal Rank Fusion, a well-studied algorithm from information retrieval research. The final merged and re-ranked list contains documents that are relevant across multiple phrasings of the question, which is a much stronger signal of genuine relevance than appearing at the top of just one list.

### The Multi-Query Prompt

The prompt is structured as a two-message ChatPromptTemplate. The system message sets the LLM's role as an assistant that generates multiple search queries. The human message provides the original query and explicitly instructs the LLM to produce exactly 4 related search queries. The OUTPUT label at the end of the human message acts as a completion cue that primes the model to produce a numbered or line-separated list of queries.

### Reciprocal Rank Fusion Explained

Reciprocal Rank Fusion takes a list of ranked document lists and computes a fused score for every unique document that appears in any of the lists.

For each document, its RRF score is the sum of 1 divided by (rank + k) across every list where it appears, where rank is the document's 0-indexed position in that list and k is a smoothing constant (default 60).

The formula 1 divided by (rank + k) means a document at rank 0 (top of a list) contributes roughly 1/60 to its score. A document at rank 10 contributes 1/70. The k constant dampens the advantage of being ranked first in any single list, preventing one very high-ranked result from dominating the fusion.

A document that appears at rank 2 in all 4 sub-query result lists accumulates a much higher fused score than a document that appears at rank 1 in only one list. This means RRF promotes documents with broad consistent relevance across multiple query formulations, which is exactly what we want.

dumps and loads from langchain_core.load are used to serialise each Document object to a JSON string for use as a dictionary key (Python dicts require hashable keys and Document objects are not hashable), and then deserialise the string back into a Document object before returning the final ranked list.

### The Full RAG Fusion Chain

The full chain connects four stages using the LCEL pipe operator.

Stage 1 generate_queries: the original question goes to the LLM which generates 4 sub-queries and splits them into a Python list.

Stage 2 retriever.map(): retriever.map() runs the retriever independently for each sub-query in the list and returns a list of 4 result lists.

Stage 3 reciprocal_rank_fusion: the 4 result lists are merged and re-scored producing a single ranked list of (Document, fused_score) tuples.

Stage 4: RunnablePassthrough passes the original question and the fused document list into the final answer prompt which is sent to Gemini for the final answer.

## With RagFusion

In [ ]:
from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatMessagePromptTemplate,
    PromptTemplate,
)

In [ ]:
# Multi-query generation prompt for RAG Fusion
# This prompt instructs the LLM to produce 4 semantically diverse search queries
# from a single original user question
#
# The system message sets the LLM role as a multi-query generator
# The human message provides the original query and explicitly requests 4 outputs
# The OUTPUT label at the end cues the model to produce a list rather than prose
prompt = ChatPromptTemplate(
    input_variables=["original_query"],
    messages=[
        # System message: define the LLM role
        SystemMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=[],
                template="You are a helpful assistant that generates multiple search queries from a single input query.",
            )
        ),
        # Human message: provide the original query and request 4 diverse sub-queries
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=["original_query"],
                template=(
                    "Generate 4 search queries related to:\n"
                    "{question}\n\n"
                    "OUTPUT (4 queries):"
                ),
            )
        ),
    ],
)

In [ ]:
# original_query stores the user question that will be passed to the generate_queries chain
original_query = "universal studios Singapore"

In [ ]:
# Build the multi-query generation chain
# prompt: formats the original question into the multi-query prompt template
# llm: Gemini generates 4 diverse sub-queries from the prompt
# StrOutputParser: extracts the plain text string from the LLM response object
# lambda x: x.split('\n'): splits the newline-separated sub-queries into a Python list
generate_queries = (
    prompt | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

In [ ]:
# Inspect the generate_queries runnable to confirm its type and structure
generate_queries

In [ ]:
from langchain_core.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    # fused_scores accumulates the RRF score for every unique document across all result lists
    # Keys are JSON-serialised document strings (documents are not hashable so we serialise them)
    # Values are the running sum of 1/(rank+k) contributions from each list
    fused_scores = {}

    for docs in results:
        # Each 'docs' is one ranked result list from one sub-query retrieval
        # We assume docs are ordered from most to least relevant (rank 0 = most relevant)
        for rank, doc in enumerate(docs):
            # Serialise the Document to a JSON string so it can be used as a dict key
            doc_str = dumps(doc)

            # Initialise the score to 0 the first time this document is seen
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0

            # Add 1/(rank+k) to this document's fused score
            # rank+k prevents division by zero and dampens the advantage of rank 0
            # A document appearing near the top of multiple lists accumulates more score
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort all documents by their accumulated fused score in descending order
    # Deserialise each document string back to a Document object before returning
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(
            fused_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]

    return reranked_results

In [ ]:
# Build the RAG Fusion retrieval chain (without the final answer LLM)
# Stage 1: generate_queries takes the original question and produces 4 sub-queries as a list
# Stage 2: retriever.map() runs the retriever once per sub-query and returns 4 result lists
# Stage 3: reciprocal_rank_fusion merges and re-scores all 4 lists into one ranked list
#
# Example data flow for 'Tell me about Universal Studios Singapore':
# After generate_queries:
#   ['Universal Studios Singapore attractions',
#    'Universal Studios Singapore history',
#    'Universal Studios Singapore rides',
#    'USS ticket prices']
#
# After retriever.map():
#   [[doc1, doc2, ...], [doc3, doc1, ...], [doc2, doc4, ...], [doc1, doc3, ...]]
#
# After reciprocal_rank_fusion:
#   [(doc1, 0.21), (doc2, 0.17), (doc3, 0.14), ...]
#   doc1 appears near the top of multiple lists so it accumulates the highest fused score
ragfusion_chain = generate_queries | retriever.map() | reciprocal_rank_fusion

### Understanding the ragfusion_chain Data Flow

The variable ragfusion_chain is a runnable pipeline object, not the results themselves. When you invoke it, for example:

results = ragfusion_chain.invoke('Tell me about Universal Studios Singapore')

you receive a reranked list of (Document, score) tuples. These top-ranked Document objects form the context that will be passed to the final answer LLM in the full_rag_fusion_chain below.

What you get after retriever.map() is a list of 4 lists, one per sub-query, each containing the documents retrieved for that specific sub-query. The reciprocal_rank_fusion function then merges all 4 lists into one unified ranking where documents that appeared consistently near the top across multiple sub-query retrievals receive the highest fused scores.

In [ ]:
# Inspect the input schema of the ragfusion_chain
# input_schema.schema() returns a JSON schema dict describing what inputs the chain expects
# This tells you the required input variable names and their data types
ragfusion_chain.input_schema.schema()

In [ ]:
# Test the ragfusion_chain on its own to see the reranked (Document, score) tuples
# before the final answer LLM is added
ragfusion_chain.invoke({
    "question": "Tell me about Singapore's nightlife scene?"
})

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Final answer prompt template for the full RAG Fusion chain
# Identical structure to the baseline chain prompt but receives reranked context
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# Build the complete end-to-end RAG Fusion chain
# context key: ragfusion_chain generates sub-queries, retrieves, and fuses results
#   the fused list of (Document, score) tuples is passed here as the context
# question key: RunnablePassthrough forwards the original question unchanged to the prompt
# The filled prompt goes to Gemini which generates the final grounded answer
# StrOutputParser extracts the plain string from the LLM output object
full_rag_fusion_chain = (
    {
        "context": ragfusion_chain,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Inspect the input schema of the full chain to confirm it expects a 'question' key
full_rag_fusion_chain.input_schema.schema()

In [ ]:
# Run the full RAG Fusion chain end-to-end
# Compare this answer against the baseline chain answer from Category 4
# RAG Fusion typically produces more complete answers because the LLM receives context
# drawn from multiple retrieval angles rather than a single query
full_rag_fusion_chain.invoke({"question": "Tell me about Singapore's nightlife scene?"})

### Sample Output

Singapore's nightlife scene can be sampled along the Singapore River, where former warehouses have been converted into top nightlife zones. There is also a bar-centric area that is home to some of the world's best watering holes, including two Singaporean bars on the World's 50 Best list. New bars and clubs open regularly across the city making Singapore's nightlife one of the most dynamic in Southeast Asia.

# Conclusion

In this notebook, RAG Fusion was implemented as an advanced retrieval technique that improves document retrieval by generating multiple search queries from a single user question. Instead of relying on a single query, the LLM generates several semantically different queries, enabling the retriever to discover more relevant information across different phrasings and aspects of the same question.

The retrieved documents from all generated queries were then combined using Reciprocal Rank Fusion, which gives higher importance to documents that consistently appear across multiple retrieval results. The formula 1 divided by (rank + k) summed across all result lists ensures that documents with broad consistent relevance are promoted above documents that ranked first in only one list. This approach improves recall, reduces the impact of query wording sensitivity, and produces a more reliable and diverse set of context documents for the LLM.

The reranked documents were passed to Gemini 2.5 Flash which generated a context-aware answer grounded strictly in the retrieved content. Compared to the standard single-query RAG baseline built in Category 4, the RAG Fusion pipeline retrieved broader and more relevant context for the same questions, resulting in more accurate and comprehensive responses for complex or ambiguous queries.

Overall this notebook demonstrates how RAG Fusion significantly enhances the retrieval stage of a RAG pipeline while remaining straightforward to implement with LangChain LCEL. It is an effective technique for building more robust, accurate, and reliable AI applications over large multi-document knowledge bases.